In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from scipy import stats

from sklearn.datasets import fetch_openml

RANDOM_STATE = 42
ALPHA = 0.05
RISK_THRESHOLD = 0.50


# Load the complete dataset
raw_data = fetch_openml(
    "airlines",
    version=1,
    as_frame=True,
    parser="auto"
)

df = raw_data.frame.copy()

print("Dataset shape:", df.shape)
print("Number of records:", len(df))

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

Dataset shape: (539383, 8)
Number of records: 539383

Columns:
['Airline', 'Flight', 'AirportFrom', 'AirportTo', 'DayOfWeek', 'Time', 'Length', 'Delay']

First 5 rows:


,Airline,Flight,AirportFrom,AirportTo,DayOfWeek,Time,Length,Delay
0,CO,269,SFO,IAH,3,15,205,1
1,US,1558,PHX,CLT,3,15,222,1
2,AA,2400,LAX,DFW,3,20,165,1
3,AA,2466,SFO,DFW,3,20,195,1
4,AS,108,ANC,SEA,3,30,202,0



Data types:
Airline        category
Flight            int64
AirportFrom    category
AirportTo      category
DayOfWeek      category
Time              int64
Length            int64
Delay          category
dtype: object

Missing values:
Airline        0
Flight         0
AirportFrom    0
AirportTo      0
DayOfWeek      0
Time           0
Length         0
Delay          0
dtype: int64


In [ ]:
# ============================================================
# PIECE 2 — DATA CLEANING & FEATURE PREPARATION
# ============================================================

# Make a copy
df = df.copy()

# ------------------------------------------------------------
# 1. Convert target to numeric
# ------------------------------------------------------------

df["Delay"] = pd.to_numeric(
    df["Delay"],
    errors="coerce"
)

# Remove rows with missing target
df = df.dropna(subset=["Delay"]).copy()

# Convert target to integer
df["Delay"] = df["Delay"].astype(int)


# ------------------------------------------------------------
# 2. Convert categorical columns to string
# ------------------------------------------------------------

categorical_columns = [
    "Airline",
    "AirportFrom",
    "AirportTo",
    "DayOfWeek"
]

for col in categorical_columns:
    df[col] = df[col].astype(str)


# ------------------------------------------------------------
# 3. Convert numerical columns
# ------------------------------------------------------------

numerical_columns = [
    "Flight",
    "Time",
    "Length"
]

for col in numerical_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )


# ------------------------------------------------------------
# 4. Check missing values
# ------------------------------------------------------------

print("Missing values after conversion:")
print(df.isnull().sum())


# ------------------------------------------------------------
# 5. Remove rows with missing feature values
# ------------------------------------------------------------

df = df.dropna(
    subset=numerical_columns + categorical_columns
).copy()

df.reset_index(drop=True, inplace=True)


# ------------------------------------------------------------
# 6. Check final dataset
# ------------------------------------------------------------

print("\nFinal dataset shape:")
print(df.shape)

print("\nData types:")
print(df.dtypes)

print("\nTarget distribution:")
print(df["Delay"].value_counts())

print("\nTarget distribution (%):")
print(
    (df["Delay"].value_counts(normalize=True) * 100)
    .round(2)
)


# ------------------------------------------------------------
# 7. Check unique values
# ------------------------------------------------------------

print("\nUnique values:")
for col in df.columns:
    print(
        f"{col:12s}: "
        f"{df[col].nunique():,}"
    )

Missing values after conversion:
Airline        0
Flight         0
AirportFrom    0
AirportTo      0
DayOfWeek      0
Time           0
Length         0
Delay          0
dtype: int64

Final dataset shape:
(539383, 8)

Data types:
Airline        object
Flight          int64
AirportFrom    object
AirportTo      object
DayOfWeek      object
Time            int64
Length          int64
Delay           int64
dtype: object

Target distribution:
Delay
0    299119
1    240264
Name: count, dtype: int64

Target distribution (%):
Delay
0    55.46
1    44.54
Name: proportion, dtype: float64

Unique values:
Airline     : 18
Flight      : 6,585
AirportFrom : 293
AirportTo   : 293
DayOfWeek   : 7
Time        : 1,131
Length      : 426
Delay       : 2


In [ ]:
# ============================================================
# PIECE 3 — TRAIN / VALIDATION / TEST SPLIT + PREPROCESSING
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.impute import SimpleImputer


# ------------------------------------------------------------
# 1. Define features
# ------------------------------------------------------------

# Flight number is treated as categorical because it is an
# identifier, not a continuous numerical quantity.

categorical_features = [
    "Airline",
    "Flight",
    "AirportFrom",
    "AirportTo",
    "DayOfWeek"
]

numerical_features = [
    "Time",
    "Length"
]

feature_columns = (
    categorical_features
    + numerical_features
)

target_column = "Delay"


# ------------------------------------------------------------
# 2. Create X and y
# ------------------------------------------------------------

X = df[feature_columns].copy()
y = df[target_column].copy()

print("Feature shape:", X.shape)
print("Target shape :", y.shape)


# ------------------------------------------------------------
# 3. First split
#
# 70% training
# 30% temporary
# ------------------------------------------------------------

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)


# ------------------------------------------------------------
# 4. Second split
#
# 15% validation
# 15% test
# ------------------------------------------------------------

X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)


# ------------------------------------------------------------
# 5. Check split sizes
# ------------------------------------------------------------

print("\nDataset split:")
print(
    f"Training   : {len(X_train):,} "
    f"({len(X_train) / len(df):.1%})"
)

print(
    f"Validation : {len(X_validation):,} "
    f"({len(X_validation) / len(df):.1%})"
)

print(
    f"Test       : {len(X_test):,} "
    f"({len(X_test) / len(df):.1%})"
)


# ------------------------------------------------------------
# 6. Check target distribution in each split
# ------------------------------------------------------------

print("\nTarget distribution by split:")

print(
    "\nTraining:"
)
print(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(
    "\nValidation:"
)
print(
    y_validation.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(
    "\nTest:"
)
print(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)


# ============================================================
# 7. NUMERICAL PREPROCESSING
# ============================================================

numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ============================================================
# 8. CATEGORICAL PREPROCESSING
# ============================================================

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)


# ============================================================
# 9. COMBINE PREPROCESSING
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)


print("\nPreprocessing pipeline created successfully.")

print("\nCategorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

Feature shape: (539383, 7)
Target shape : (539383,)

Dataset split:
Training   : 377,568 (70.0%)
Validation : 80,907 (15.0%)
Test       : 80,908 (15.0%)

Target distribution by split:

Training:
Delay
0    55.46
1    44.54
Name: proportion, dtype: float64

Validation:
Delay
0    55.46
1    44.54
Name: proportion, dtype: float64

Test:
Delay
0    55.46
1    44.54
Name: proportion, dtype: float64

Preprocessing pipeline created successfully.

Categorical features:
['Airline', 'Flight', 'AirportFrom', 'AirportTo', 'DayOfWeek']

Numerical features:
['Time', 'Length']


In [ ]:
# ============================================================
# PIECE 4 — MODEL TRAINING & VALIDATION
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# ============================================================
# 1. EVALUATION FUNCTION
# ============================================================

def evaluate_model(
    model,
    X_data,
    y_data,
    model_name,
    threshold=0.50
):

    # Probability of Delay = 1
    probabilities = model.predict_proba(
        X_data
    )[:, 1]

    # Convert probability to class
    predictions = (
        probabilities >= threshold
    ).astype(int)

    # Metrics
    roc_auc = roc_auc_score(
        y_data,
        probabilities
    )

    pr_auc = average_precision_score(
        y_data,
        probabilities
    )

    precision = precision_score(
        y_data,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_data,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_data,
        predictions,
        zero_division=0
    )

    cm = confusion_matrix(
        y_data,
        predictions
    )

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    print(
        f"ROC-AUC   : {roc_auc:.4f}"
    )

    print(
        f"PR-AUC    : {pr_auc:.4f}"
    )

    print(
        f"Precision : {precision:.4f}"
    )

    print(
        f"Recall    : {recall:.4f}"
    )

    print(
        f"F1 Score  : {f1:.4f}"
    )

    print("\nConfusion Matrix:")
    print(cm)

    return {
        "model": model_name,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


# ============================================================
# 2. DUMMY BASELINE
# ============================================================

print("\nTraining Dummy Classifier...")

dummy_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            DummyClassifier(
                strategy="most_frequent"
            )
        )
    ]
)

dummy_model.fit(
    X_train,
    y_train
)


# ============================================================
# 3. LOGISTIC REGRESSION
# ============================================================

print("Training Logistic Regression...")

logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=500,
                class_weight="balanced",
                random_state=RANDOM_STATE
            )
        )
    ]
)

logistic_model.fit(
    X_train,
    y_train
)


# ============================================================
# 4. RANDOM FOREST
# ============================================================

print("Training Random Forest...")

random_forest_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=15,
                min_samples_leaf=5,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ]
)

random_forest_model.fit(
    X_train,
    y_train
)


# ============================================================
# 5. VALIDATION — DUMMY
# ============================================================

dummy_validation = evaluate_model(
    dummy_model,
    X_validation,
    y_validation,
    "Dummy Classifier — Validation"
)


# ============================================================
# 6. VALIDATION — LOGISTIC REGRESSION
# ============================================================

logistic_validation = evaluate_model(
    logistic_model,
    X_validation,
    y_validation,
    "Logistic Regression — Validation"
)


# ============================================================
# 7. VALIDATION — RANDOM FOREST
# ============================================================

rf_validation = evaluate_model(
    random_forest_model,
    X_validation,
    y_validation,
    "Random Forest — Validation"
)


# ============================================================
# 8. COMPARISON TABLE
# ============================================================

validation_results = pd.DataFrame([
    dummy_validation,
    logistic_validation,
    rf_validation
])

# Remove anything that isn't a metric
validation_results = validation_results[
    [
        "model",
        "roc_auc",
        "pr_auc",
        "precision",
        "recall",
        "f1"
    ]
]

print("\n")
print("=" * 70)
print("VALIDATION MODEL COMPARISON")
print("=" * 70)

display(
    validation_results.round(4)
)


Training Dummy Classifier...
Training Logistic Regression...
Training Random Forest...

Dummy Classifier — Validation
ROC-AUC   : 0.5000
PR-AUC    : 0.4454
Precision : 0.0000
Recall    : 0.0000
F1 Score  : 0.0000

Confusion Matrix:
[[44868     0]
 [36039     0]]

Logistic Regression — Validation
ROC-AUC   : 0.6986
PR-AUC    : 0.6556
Precision : 0.5991
Recall    : 0.6187
F1 Score  : 0.6087

Confusion Matrix:
[[29944 14924]
 [13741 22298]]

Random Forest — Validation
ROC-AUC   : 0.6875
PR-AUC    : 0.6429
Precision : 0.6024
Recall    : 0.5657
F1 Score  : 0.5835

Confusion Matrix:
[[31413 13455]
 [15651 20388]]


VALIDATION MODEL COMPARISON


,model,roc_auc,pr_auc,precision,recall,f1
0,Dummy Classifier — Validation,0.5000,0.4454,0.0000,0.0000,0.0000
1,Logistic Regression — Validation,0.6986,0.6556,0.5991,0.6187,0.6087
2,Random Forest — Validation,0.6875,0.6429,0.6024,0.5657,0.5835


In [ ]:
# ============================================================
# PIECE 5 — FINAL TEST EVALUATION
# ============================================================

# ------------------------------------------------------------
# 1. Evaluate Logistic Regression on untouched test data
# ------------------------------------------------------------

logistic_test = evaluate_model(
    logistic_model,
    X_test,
    y_test,
    "Logistic Regression — Final Test"
)


# ------------------------------------------------------------
# 2. Display final test metrics
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL MODEL PERFORMANCE")
print("=" * 70)

print(
    f"ROC-AUC   : {logistic_test['roc_auc']:.4f}"
)

print(
    f"PR-AUC    : {logistic_test['pr_auc']:.4f}"
)

print(
    f"Precision : {logistic_test['precision']:.4f}"
)

print(
    f"Recall    : {logistic_test['recall']:.4f}"
)

print(
    f"F1 Score  : {logistic_test['f1']:.4f}"
)


# ------------------------------------------------------------
# 3. Compare validation vs test
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "Precision",
        "Recall",
        "F1"
    ],
    "Validation": [
        logistic_validation["roc_auc"],
        logistic_validation["pr_auc"],
        logistic_validation["precision"],
        logistic_validation["recall"],
        logistic_validation["f1"]
    ],
    "Test": [
        logistic_test["roc_auc"],
        logistic_test["pr_auc"],
        logistic_test["precision"],
        logistic_test["recall"],
        logistic_test["f1"]
    ]
})

print("\nValidation vs Test:")
display(
    comparison.round(4)
)


Logistic Regression — Final Test
ROC-AUC   : 0.6978
PR-AUC    : 0.6553
Precision : 0.5963
Recall    : 0.6167
F1 Score  : 0.6063

Confusion Matrix:
[[29819 15049]
 [13814 22226]]

FINAL MODEL PERFORMANCE
ROC-AUC   : 0.6978
PR-AUC    : 0.6553
Precision : 0.5963
Recall    : 0.6167
F1 Score  : 0.6063

Validation vs Test:


,Metric,Validation,Test
0,ROC-AUC,0.6986,0.6978
1,PR-AUC,0.6556,0.6553
2,Precision,0.5991,0.5963
3,Recall,0.6187,0.6167
4,F1,0.6087,0.6063


In [ ]:
# ============================================================
# PIECE 6A — GENERATE BASELINE & PRODUCTION PREDICTIONS
# ============================================================

# ------------------------------------------------------------
# Baseline predictions
# ------------------------------------------------------------

baseline_probabilities = logistic_model.predict_proba(
    X_train
)[:, 1]


# ------------------------------------------------------------
# Production/test predictions
# ------------------------------------------------------------

production_probabilities = logistic_model.predict_proba(
    X_test
)[:, 1]


# ------------------------------------------------------------
# Convert probabilities to high-risk flags
# ------------------------------------------------------------

RISK_THRESHOLD = 0.50

baseline_risk_flags = (
    baseline_probabilities >= RISK_THRESHOLD
).astype(int)

production_risk_flags = (
    production_probabilities >= RISK_THRESHOLD
).astype(int)


# ------------------------------------------------------------
# Basic rates
# ------------------------------------------------------------

baseline_delay_rate = y_train.mean()
production_delay_rate = y_test.mean()

baseline_risk_rate = baseline_risk_flags.mean()
production_risk_rate = production_risk_flags.mean()


print("=" * 70)
print("BASELINE VS PRODUCTION WINDOW")
print("=" * 70)

print(
    f"\nBaseline records   : {len(X_train):,}"
)

print(
    f"Production records : {len(X_test):,}"
)

print(
    f"\nBaseline delay rate   : "
    f"{baseline_delay_rate:.4%}"
)

print(
    f"Production delay rate : "
    f"{production_delay_rate:.4%}"
)

print(
    f"\nBaseline high-risk rate   : "
    f"{baseline_risk_rate:.4%}"
)

print(
    f"Production high-risk rate : "
    f"{production_risk_rate:.4%}"
)

BASELINE VS PRODUCTION WINDOW

Baseline records   : 377,568
Production records : 80,908

Baseline delay rate   : 44.5443%
Production delay rate : 44.5444%

Baseline high-risk rate   : 46.0532%
Production high-risk rate : 46.0708%


In [ ]:
# ============================================================
# PIECE 6B — KS TEST FOR CONTINUOUS FEATURES
# ============================================================

from scipy.stats import ks_2samp


def ks_drift_test(
    baseline,
    production,
    feature,
    alpha=0.05
):

    baseline_values = (
        pd.to_numeric(
            baseline[feature],
            errors="coerce"
        )
        .dropna()
    )

    production_values = (
        pd.to_numeric(
            production[feature],
            errors="coerce"
        )
        .dropna()
    )

    statistic, p_value = ks_2samp(
        baseline_values,
        production_values
    )

    return {
        "Feature": feature,
        "KS Statistic": statistic,
        "p-value": p_value,
        "Drift": p_value < alpha
    }


# ------------------------------------------------------------
# Run KS test
# ------------------------------------------------------------

continuous_features = [
    "Time",
    "Length"
]

ks_results = []

for feature in continuous_features:

    result = ks_drift_test(
        X_train,
        X_test,
        feature
    )

    ks_results.append(result)


ks_results_df = pd.DataFrame(
    ks_results
)


print("=" * 70)
print("KS TEST — CONTINUOUS FEATURE DRIFT")
print("=" * 70)

display(
    ks_results_df.round(6)
)

KS TEST — CONTINUOUS FEATURE DRIFT


,Feature,KS Statistic,p-value,Drift
0,Time,0.002831,0.658469,False
1,Length,0.003422,0.415414,False


In [ ]:
# ============================================================
# UPDATED CHI-SQUARE TEST WITH PRECISION BYPASS
# ============================================================

from scipy.stats import chisquare

def chi_square_drift_test(
    baseline,
    production,
    feature,
    alpha=0.05
):

    # Count categories
    baseline_counts = (
        baseline[feature]
        .value_counts()
    )

    production_counts = (
        production[feature]
        .value_counts()
    )

    # Include categories from both datasets
    categories = sorted(
        set(baseline_counts.index)
        | set(production_counts.index)
    )

    # Observed production counts
    observed = np.array([
        production_counts.get(
            category,
            0
        )
        for category in categories
    ], dtype=float)

    # Baseline proportions
    baseline_proportions = (
        baseline_counts
        / baseline_counts.sum()
    )

    # Expected production counts based on baseline distribution
    expected = np.array([
        baseline_proportions.get(
            category,
            0
        ) for category in categories
    ], dtype=float)

    # Robust normalization to match sums
    total_observed = np.sum(observed)
    expected = expected * (total_observed / np.sum(expected))

    # Handle edge case where normalization might still have tiny epsilon errors
    # by adjusting the largest category to make the sum exact.
    diff = total_observed - np.sum(expected)
    if diff != 0:
        expected[np.argmax(expected)] += diff

    # Remove zero expected values to avoid division by zero
    valid = expected > 0
    observed = observed[valid]
    expected = expected[valid]

    # Chi-square test
    # FIX: Use sum_check=False to bypass strict precision errors in SciPy
    statistic, p_value = chisquare(
        f_obs=observed,
        f_exp=expected,
        sum_check=False
    )

    # Cohen's w — effect size
    effect_size = np.sqrt(
        statistic / total_observed
    )

    return {
        "Feature": feature,
        "Chi-Square Statistic": statistic,
        "p-value": p_value,
        "Effect Size": effect_size,
        "Drift": p_value < alpha
    }

# ------------------------------------------------------------
# Run Chi-square tests
# ------------------------------------------------------------

CATEGORICAL_EFFECT_THRESHOLD = 0.10

chi_alerts = []
chi_square_results = []

# Flight is excluded from drift monitoring because it has
# 6,585 categories and is primarily an identifier.
drift_categorical_features = [
    "Airline",
    "AirportFrom",
    "AirportTo",
    "DayOfWeek"
]

for feature in drift_categorical_features:

    result = chi_square_drift_test(
        X_train,
        X_test,
        feature
    )

    statistically_significant = (
        result["p-value"] < ALPHA
    )

    practically_significant = (
        result["Effect Size"]
        >= CATEGORICAL_EFFECT_THRESHOLD
    )

    alert = (
        statistically_significant
        and
        practically_significant
    )

    chi_alerts.append({
        "Feature": result["Feature"],
        "Chi-Square Statistic":
            result["Chi-Square Statistic"],
        "p-value":
            result["p-value"],
        "Effect Size":
            result["Effect Size"],
        "Alert": alert
    })

    chi_square_results.append({
        "Feature": result["Feature"],
        "Chi-Square Statistic":
            result["Chi-Square Statistic"],
        "p-value":
            result["p-value"],
        "Drift":
            result["Drift"]
    })


chi_alerts_df = pd.DataFrame(
    chi_alerts
)

print("=" * 70)
print("CATEGORICAL DRIFT — CHI-SQUARE + EFFECT SIZE")
print("=" * 70)

display(
    chi_alerts_df.round(6)
)

CATEGORICAL DRIFT — CHI-SQUARE + EFFECT SIZE


,Feature,Chi-Square Statistic,p-value,Effect Size,Alert
0,Airline,27.989670,0.045059,0.018600,False
1,AirportFrom,363.681748,0.002715,0.067045,False
2,AirportTo,354.297540,0.007311,0.066174,False
3,DayOfWeek,4.586666,0.597808,0.007529,False


In [ ]:
# ============================================================
# PIECE 6D — TWO-PROPORTION Z-TEST
# ============================================================

from scipy.stats import norm


def two_proportion_z_test(
    baseline_values,
    production_values,
    alpha=0.05
):

    baseline_values = np.asarray(
        baseline_values
    ).astype(int)

    production_values = np.asarray(
        production_values
    ).astype(int)

    # Number of successes
    x1 = baseline_values.sum()
    x2 = production_values.sum()

    # Sample sizes
    n1 = len(baseline_values)
    n2 = len(production_values)

    # Individual proportions
    p1 = x1 / n1
    p2 = x2 / n2

    # Pooled proportion
    pooled_p = (
        x1 + x2
    ) / (
        n1 + n2
    )

    # Standard error
    standard_error = np.sqrt(
        pooled_p
        * (1 - pooled_p)
        * (
            (1 / n1)
            + (1 / n2)
        )
    )

    # Z statistic
    z_statistic = (
        p2 - p1
    ) / standard_error

    # Two-sided p-value
    p_value = 2 * (
        1 - norm.cdf(
            abs(z_statistic)
        )
    )

    return {
        "Baseline Rate": p1,
        "Production Rate": p2,
        "Rate Difference": p2 - p1,
        "Z Statistic": z_statistic,
        "p-value": p_value,
        "Drift": p_value < alpha
    }


# ============================================================
# TEST 1 — ACTUAL DELAY RATE
# ============================================================

delay_rate_result = two_proportion_z_test(
    y_train.values,
    y_test.values
)


print("=" * 70)
print("Z-TEST — ACTUAL DELAY RATE")
print("=" * 70)

for key, value in delay_rate_result.items():
    print(f"{key:20s}: {value}")


# ============================================================
# TEST 2 — MODEL HIGH-RISK RATE
# ============================================================

risk_rate_result = two_proportion_z_test(
    baseline_risk_flags,
    production_risk_flags
)


print("\n" + "=" * 70)
print("Z-TEST — MODEL HIGH-RISK RATE")
print("=" * 70)

for key, value in risk_rate_result.items():
    print(f"{key:20s}: {value}")

Z-TEST — ACTUAL DELAY RATE
Baseline Rate       : 0.4454429400796678
Production Rate     : 0.44544420823651554
Rate Difference     : 1.2681568477557192e-06
Z Statistic         : 0.0006586259071371657
p-value             : 0.9994744925953436
Drift               : False

Z-TEST — MODEL HIGH-RISK RATE
Baseline Rate       : 0.4605316128485465
Production Rate     : 0.4607084589904583
Rate Difference     : 0.0001768461419118239
Z Statistic         : 0.0915831828986041
p-value             : 0.9270292133924913
Drift               : False


In [ ]:
# ============================================================
# PIECE 7 — CONTROLLED DRIFT INJECTION
# ============================================================

drift_levels = [
    0.00,
    0.05,
    0.10,
    0.20,
    0.30
]

drift_results = []


for drift_level in drift_levels:

    # Copy the complete test window
    simulated_data = X_test.copy()

    # Inject drift into Length
    simulated_data["Length"] = (
        simulated_data["Length"]
        * (1 + drift_level)
    )

    # Run KS test
    result = ks_drift_test(
        X_train,
        simulated_data,
        "Length"
    )

    drift_results.append({
        "Injected Shift": drift_level,
        "KS Statistic": result["KS Statistic"],
        "p-value": result["p-value"],
        "Drift Detected": result["Drift"]
    })


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

drift_results_df = pd.DataFrame(
    drift_results
)

print("=" * 70)
print("CONTROLLED DRIFT INJECTION — LENGTH")
print("=" * 70)

display(
    drift_results_df.round(6)
)

CONTROLLED DRIFT INJECTION — LENGTH


,Injected Shift,KS Statistic,p-value,Drift Detected
0,0.00,0.003422,0.415414,False
1,0.05,0.052768,0.000000,True
2,0.10,0.091115,0.000000,True
3,0.20,0.154236,0.000000,True
4,0.30,0.213208,0.000000,True


In [ ]:
# ============================================================
# PIECE 8 — CONTROLLED CATEGORICAL DRIFT INJECTION
# ============================================================

def inject_airline_drift(
    data,
    airline,
    shift_fraction
):

    data = data.copy()

    # All rows that are NOT the selected airline
    other_indices = data.index[
        data["Airline"] != airline
    ]

    # Number of records to reassign
    n_to_change = int(
        len(data) * shift_fraction
    )

    n_to_change = min(
        n_to_change,
        len(other_indices)
    )

    rng = np.random.default_rng(
        RANDOM_STATE
    )

    selected_indices = rng.choice(
        other_indices,
        size=n_to_change,
        replace=False
    )

    # Change selected records to the
    # chosen airline
    data.loc[
        selected_indices,
        "Airline"
    ] = airline

    return data


# ============================================================
# Select the most common airline
# ============================================================

most_common_airline = (
    X_test["Airline"]
    .value_counts()
    .index[0]
)

print(
    "Airline selected for drift injection:",
    most_common_airline
)


# ============================================================
# Drift levels
# ============================================================

categorical_drift_levels = [
    0.00,
    0.05,
    0.10,
    0.20,
    0.30
]


categorical_drift_results = []


# ============================================================
# Run experiment
# ============================================================

for drift_level in categorical_drift_levels:

    # Start from ORIGINAL production/test data
    original_data = X_test.copy()

    # Inject artificial drift
    simulated_data = inject_airline_drift(
        original_data,
        most_common_airline,
        drift_level
    )

    # IMPORTANT:
    # Compare original production distribution
    # against modified production distribution.
    result = chi_square_drift_test(
        original_data,
        simulated_data,
        "Airline"
    )

    categorical_drift_results.append({
        "Injected Shift": drift_level,
        "Chi-Square Statistic": result[
            "Chi-Square Statistic"
        ],
        "p-value": result["p-value"],
        "Drift Detected": result["Drift"]
    })


# ============================================================
# Display results
# ============================================================

categorical_drift_results_df = pd.DataFrame(
    categorical_drift_results
)

print("\n" + "=" * 70)
print("CONTROLLED CATEGORICAL DRIFT — AIRLINE")
print("=" * 70)

display(
    categorical_drift_results_df.round(6)
)

Airline selected for drift injection: WN

CONTROLLED CATEGORICAL DRIFT — AIRLINE


,Injected Shift,Chi-Square Statistic,p-value,Drift Detected
0,0.00,0.000000,1.0,False
1,0.05,1406.532695,0.0,True
2,0.10,5626.621226,0.0,True
3,0.20,22503.136787,0.0,True
4,0.30,50633.462595,0.0,True


In [ ]:
# ============================================================
# PIECE 9 — CONTROLLED TARGET DRIFT
# TWO-PROPORTION Z-TEST
# ============================================================

def inject_delay_rate(
    labels,
    target_rate,
    random_state=42
):

    labels = np.asarray(labels).copy()

    n = len(labels)
    target_count = int(
        n * target_rate
    )

    rng = np.random.default_rng(
        random_state
    )

    # Create labels with desired proportion
    new_labels = np.zeros(n, dtype=int)

    selected_indices = rng.choice(
        n,
        size=target_count,
        replace=False
    )

    new_labels[selected_indices] = 1

    return new_labels


# ============================================================
# Baseline = ORIGINAL TEST distribution
# ============================================================

baseline_labels = y_test.values.copy()

baseline_rate = baseline_labels.mean()

print(
    f"Original test delay rate: "
    f"{baseline_rate:.4%}"
)


# ============================================================
# Target delay rates to simulate
# ============================================================

target_rates = [
    baseline_rate,
    0.40,
    0.50,
    0.55,
    0.60
]


z_test_results = []


# ============================================================
# Run experiment
# ============================================================

for target_rate in target_rates:

    simulated_labels = inject_delay_rate(
        baseline_labels,
        target_rate,
        RANDOM_STATE
    )

    result = two_proportion_z_test(
        baseline_labels,
        simulated_labels
    )

    z_test_results.append({
        "Target Delay Rate": target_rate,
        "Actual Simulated Rate":
            simulated_labels.mean(),
        "Rate Difference":
            result["Rate Difference"],
        "Z Statistic":
            result["Z Statistic"],
        "p-value":
            result["p-value"],
        "Drift Detected":
            result["Drift"]
    })


# ============================================================
# Display results
# ============================================================

z_test_results_df = pd.DataFrame(
    z_test_results
)

print("\n" + "=" * 70)
print("CONTROLLED TARGET DRIFT — TWO-PROPORTION Z-TEST")
print("=" * 70)

display(
    z_test_results_df.round(6)
)

Original test delay rate: 44.5444%

CONTROLLED TARGET DRIFT — TWO-PROPORTION Z-TEST


,Target Delay Rate,Actual Simulated Rate,Rate Difference,Z Statistic,p-value,Drift Detected
0,0.445444,0.445444,0.000000,0.000000,1.0,False
1,0.400000,0.399998,-0.045447,-18.503894,0.0,True
2,0.500000,0.500000,0.054556,21.978541,0.0,True
3,0.550000,0.549995,0.104551,42.057437,0.0,True
4,0.600000,0.599990,0.154546,62.232457,0.0,True


In [ ]:
# ============================================================
# PIECE 10 — FINAL DRIFT ALERT ENGINE
# ============================================================

# ------------------------------------------------------------
# Practical thresholds
# ------------------------------------------------------------

KS_THRESHOLD = 0.05

# Minimum absolute change in delay/risk rate
RATE_THRESHOLD = 0.02


# ------------------------------------------------------------
# 1. Evaluate continuous drift
# ------------------------------------------------------------

ks_alerts = []

for result in ks_results:

    statistically_significant = (
        result["p-value"] < ALPHA
    )

    practically_significant = (
        result["KS Statistic"] >= KS_THRESHOLD
    )

    alert = (
        statistically_significant
        and practically_significant
    )

    ks_alerts.append({
        "Feature": result["Feature"],
        "KS Statistic": result["KS Statistic"],
        "p-value": result["p-value"],
        "Alert": alert
    })


ks_alerts_df = pd.DataFrame(
    ks_alerts
)


# ------------------------------------------------------------
# 2. Categorical drift
# ------------------------------------------------------------
#
# chi_alerts_df was already calculated in Piece 7 using:
#
# p-value < 0.05
# AND
# Effect Size >= 0.10
#
# Do NOT recreate it using p-value alone.
# ------------------------------------------------------------

categorical_alert = (
    chi_alerts_df["Alert"].any()
)


# ------------------------------------------------------------
# 3. Actual delay-rate alert
# ------------------------------------------------------------

delay_rate_difference = abs(
    delay_rate_result["Rate Difference"]
)

delay_rate_alert = (
    delay_rate_result["p-value"] < ALPHA
    and
    delay_rate_difference >= RATE_THRESHOLD
)


# ------------------------------------------------------------
# 4. Model risk-rate alert
# ------------------------------------------------------------

risk_rate_difference = abs(
    risk_rate_result["Rate Difference"]
)

risk_rate_alert = (
    risk_rate_result["p-value"] < ALPHA
    and
    risk_rate_difference >= RATE_THRESHOLD
)


# ============================================================
# 5. FINAL ALERT
# ============================================================

continuous_alert = (
    ks_alerts_df["Alert"].any()
)

feature_alert = (
    continuous_alert
    or
    categorical_alert
)

overall_alert = (
    feature_alert
    or
    delay_rate_alert
    or
    risk_rate_alert
)


# ============================================================
# 6. FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("FINAL AVIATION OPERATIONAL STATUS")
print("=" * 70)


if overall_alert:

    print("\n🚨 DRIFT ALERT")

    print(
        "Practically significant operational "
        "drift detected."
    )

    print("\nRecommended action:")

    print(
        "1. Investigate changed flight patterns."
    )

    print(
        "2. Review model performance."
    )

    print(
        "3. Consider model retraining."
    )

else:

    print("\n✓ SYSTEM STABLE")

    print(
        "No practically significant "
        "operational drift detected."
    )


FINAL AVIATION OPERATIONAL STATUS

✓ SYSTEM STABLE
No practically significant operational drift detected.


In [ ]:
# ============================================================
# PIECE 11 — MODEL EXPLAINABILITY
# LOGISTIC REGRESSION COEFFICIENT ANALYSIS
# ============================================================

# ------------------------------------------------------------
# 1. Extract the fitted preprocessing pipeline
# ------------------------------------------------------------

fitted_preprocessor = (
    logistic_model
    .named_steps["preprocessor"]
)

fitted_classifier = (
    logistic_model
    .named_steps["classifier"]
)


# ------------------------------------------------------------
# 2. Get transformed feature names
# ------------------------------------------------------------

feature_names = (
    fitted_preprocessor
    .get_feature_names_out()
)


# ------------------------------------------------------------
# 3. Get Logistic Regression coefficients
# ------------------------------------------------------------

coefficients = (
    fitted_classifier
    .coef_[0]
)


# ------------------------------------------------------------
# 4. Create feature-importance table
# ------------------------------------------------------------

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Absolute Importance": np.abs(
        coefficients
    )
})


# ------------------------------------------------------------
# 5. Sort by importance
# ------------------------------------------------------------

feature_importance = (
    feature_importance
    .sort_values(
        "Absolute Importance",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Display top 20 features
# ------------------------------------------------------------

print("=" * 70)
print("TOP FEATURES ASSOCIATED WITH FLIGHT DELAY RISK")
print("=" * 70)

display(
    feature_importance.head(20).round(4)
)

TOP FEATURES ASSOCIATED WITH FLIGHT DELAY RISK


,Feature,Coefficient,Absolute Importance
0,categorical__Flight_4815,2.1804,2.1804
1,categorical__Flight_3351,1.8259,1.8259
2,categorical__Flight_4754,-1.6546,1.6546
3,categorical__Flight_4607,1.5785,1.5785
4,categorical__Flight_3133,-1.5652,1.5652
5,categorical__Flight_1963,1.5276,1.5276
6,categorical__Flight_5055,-1.5155,1.5155
7,categorical__Flight_1873,1.4941,1.4941
8,categorical__Flight_971,-1.4923,1.4923
9,categorical__Flight_4279,-1.4880,1.4880


In [ ]:
# ============================================================
# POSITIVE DELAY-RISK FEATURES
# ============================================================

positive_features = (
    feature_importance[
        feature_importance["Coefficient"] > 0
    ]
    .sort_values(
        "Coefficient",
        ascending=False
    )
)

print("=" * 70)
print("FEATURES ASSOCIATED WITH HIGHER DELAY RISK")
print("=" * 70)

display(
    positive_features.head(15).round(4)
)


# ============================================================
# NEGATIVE DELAY-RISK FEATURES
# ============================================================

negative_features = (
    feature_importance[
        feature_importance["Coefficient"] < 0
    ]
    .sort_values(
        "Coefficient",
        ascending=True
    )
)

print("=" * 70)
print("FEATURES ASSOCIATED WITH LOWER DELAY RISK")
print("=" * 70)

display(
    negative_features.head(15).round(4)
)

FEATURES ASSOCIATED WITH HIGHER DELAY RISK


,Feature,Coefficient,Absolute Importance
0,categorical__Flight_4815,2.1804,2.1804
1,categorical__Flight_3351,1.8259,1.8259
3,categorical__Flight_4607,1.5785,1.5785
5,categorical__Flight_1963,1.5276,1.5276
7,categorical__Flight_1873,1.4941,1.4941
13,categorical__Flight_1374,1.4325,1.4325
14,categorical__Airline_WN,1.4162,1.4162
16,categorical__Flight_3517,1.4053,1.4053
17,categorical__Flight_5519,1.3967,1.3967
19,categorical__Flight_3631,1.3906,1.3906


FEATURES ASSOCIATED WITH LOWER DELAY RISK


,Feature,Coefficient,Absolute Importance
2,categorical__Flight_4754,-1.6546,1.6546
4,categorical__Flight_3133,-1.5652,1.5652
6,categorical__Flight_5055,-1.5155,1.5155
8,categorical__Flight_971,-1.4923,1.4923
9,categorical__Flight_4279,-1.4880,1.4880
10,categorical__Flight_2761,-1.4611,1.4611
11,categorical__Flight_2113,-1.4432,1.4432
12,categorical__Flight_4635,-1.4379,1.4379
15,categorical__Flight_3301,-1.4108,1.4108
18,categorical__Flight_3243,-1.3949,1.3949


In [ ]:
# ============================================================
# PIECE 12 — OPERATIONAL FEATURE IMPORTANCE
# ============================================================

# ------------------------------------------------------------
# 1. Extract coefficients from the trained Logistic Regression
# ------------------------------------------------------------

coefficients = (
    logistic_model
    .named_steps["classifier"]
    .coef_[0]
)

feature_names = (
    logistic_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)


# ------------------------------------------------------------
# 2. Create coefficient dataframe
# ------------------------------------------------------------

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})


# ------------------------------------------------------------
# 3. Remove Flight features
# ------------------------------------------------------------

operational_coef_df = coef_df[
    ~coef_df["Feature"].str.contains(
        "Flight_"
    )
].copy()


# ------------------------------------------------------------
# 4. Calculate absolute importance
# ------------------------------------------------------------

operational_coef_df[
    "Absolute Importance"
] = np.abs(
    operational_coef_df["Coefficient"]
)


# ------------------------------------------------------------
# 5. Sort by importance
# ------------------------------------------------------------

operational_coef_df = (
    operational_coef_df
    .sort_values(
        "Absolute Importance",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Display top operational drivers
# ------------------------------------------------------------

print("=" * 70)
print("TOP OPERATIONAL FEATURES ASSOCIATED WITH DELAY RISK")
print("=" * 70)

display(
    operational_coef_df
    .head(20)
    .round(4)
)

TOP OPERATIONAL FEATURES ASSOCIATED WITH DELAY RISK


,Feature,Coefficient,Absolute Importance
0,categorical__Airline_WN,1.4162,1.4162
1,categorical__AirportTo_OME,1.2983,1.2983
2,categorical__AirportFrom_IYK,-1.0494,1.0494
3,categorical__AirportTo_OTZ,1.0385,1.0385
4,categorical__AirportTo_STX,0.9781,0.9781
5,categorical__AirportTo_LWS,-0.9070,0.9070
6,categorical__AirportFrom_LWS,0.9044,0.9044
7,categorical__AirportTo_PAH,-0.9015,0.9015
8,categorical__AirportTo_JAC,0.8892,0.8892
9,categorical__AirportFrom_ORD,0.8828,0.8828


In [ ]:
# ============================================================
# HIGHER DELAY-RISK OPERATIONAL FEATURES
# ============================================================

higher_risk_features = (
    operational_coef_df[
        operational_coef_df["Coefficient"] > 0
    ]
    .sort_values(
        "Coefficient",
        ascending=False
    )
)

print("=" * 70)
print("HIGHER DELAY-RISK OPERATIONAL FEATURES")
print("=" * 70)

display(
    higher_risk_features
    .head(15)
    .round(4)
)


# ============================================================
# LOWER DELAY-RISK OPERATIONAL FEATURES
# ============================================================

lower_risk_features = (
    operational_coef_df[
        operational_coef_df["Coefficient"] < 0
    ]
    .sort_values(
        "Coefficient",
        ascending=True
    )
)

print("=" * 70)
print("LOWER DELAY-RISK OPERATIONAL FEATURES")
print("=" * 70)

display(
    lower_risk_features
    .head(15)
    .round(4)
)

HIGHER DELAY-RISK OPERATIONAL FEATURES


,Feature,Coefficient,Absolute Importance
0,categorical__Airline_WN,1.4162,1.4162
1,categorical__AirportTo_OME,1.2983,1.2983
3,categorical__AirportTo_OTZ,1.0385,1.0385
4,categorical__AirportTo_STX,0.9781,0.9781
6,categorical__AirportFrom_LWS,0.9044,0.9044
8,categorical__AirportTo_JAC,0.8892,0.8892
9,categorical__AirportFrom_ORD,0.8828,0.8828
13,categorical__AirportTo_ADQ,0.8690,0.8690
14,categorical__AirportFrom_STX,0.8410,0.8410
20,categorical__AirportTo_SJU,0.7416,0.7416


LOWER DELAY-RISK OPERATIONAL FEATURES


,Feature,Coefficient,Absolute Importance
2,categorical__AirportFrom_IYK,-1.0494,1.0494
5,categorical__AirportTo_LWS,-0.9070,0.9070
7,categorical__AirportTo_PAH,-0.9015,0.9015
10,categorical__AirportTo_BGM,-0.8822,0.8822
11,categorical__AirportFrom_YAK,-0.8784,0.8784
12,categorical__AirportFrom_KOA,-0.8690,0.8690
15,categorical__AirportFrom_ECP,-0.8369,0.8369
16,categorical__AirportFrom_LFT,-0.8260,0.8260
17,categorical__AirportTo_BGR,-0.7939,0.7939
18,categorical__AirportTo_TXK,-0.7696,0.7696


In [ ]:
# ============================================================
# PIECE 13 — RISK SEGMENTATION & CALIBRATION CHECK
# ============================================================

# Get test-set predicted probabilities
test_probabilities = logistic_model.predict_proba(
    X_test
)[:, 1]

risk_analysis = X_test.copy()

risk_analysis["Actual Delay"] = y_test.values

risk_analysis["Predicted Risk"] = test_probabilities


# ------------------------------------------------------------
# Create risk segments
# ------------------------------------------------------------

risk_analysis["Risk Segment"] = pd.cut(
    risk_analysis["Predicted Risk"],
    bins=[
        -np.inf,
        0.30,
        0.50,
        0.70,
        np.inf
    ],
    labels=[
        "Low",
        "Moderate",
        "High",
        "Very High"
    ]
)


# ------------------------------------------------------------
# Calculate segment statistics
# ------------------------------------------------------------

risk_summary = (
    risk_analysis
    .groupby(
        "Risk Segment",
        observed=True
    )
    .agg(
        Flights=("Actual Delay", "size"),
        Actual_Delay_Rate=("Actual Delay", "mean"),
        Average_Predicted_Risk=("Predicted Risk", "mean")
    )
    .reset_index()
)


risk_summary["Actual_Delay_Rate"] = (
    risk_summary["Actual_Delay_Rate"] * 100
)

risk_summary["Average_Predicted_Risk"] = (
    risk_summary["Average_Predicted_Risk"] * 100
)


print("=" * 70)
print("FLIGHT DELAY RISK SEGMENTATION")
print("=" * 70)

display(
    risk_summary.round(2)
)

FLIGHT DELAY RISK SEGMENTATION


,Risk Segment,Flights,Actual_Delay_Rate,Average_Predicted_Risk
0,Low,13738,22.05,22.90
1,Moderate,29895,36.08,40.28
2,High,24111,51.35,59.16
3,Very High,13164,74.78,78.85


In [ ]:
# ============================================================
# PIECE 14 — PROBABILITY CALIBRATION
# ============================================================

from sklearn.metrics import (
    brier_score_loss
)

# ------------------------------------------------------------
# 1. Brier Score
# ------------------------------------------------------------

brier_score = brier_score_loss(
    y_test,
    test_probabilities
)

print("=" * 70)
print("MODEL PROBABILITY CALIBRATION")
print("=" * 70)

print(
    f"Brier Score: {brier_score:.4f}"
)


# ------------------------------------------------------------
# 2. Calibration table
# ------------------------------------------------------------

calibration_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted Probability": test_probabilities
})

calibration_df["Risk Bin"] = pd.qcut(
    calibration_df["Predicted Probability"],
    q=10,
    duplicates="drop"
)

calibration_summary = (
    calibration_df
    .groupby(
        "Risk Bin",
        observed=True
    )
    .agg(
        Flights=("Actual", "size"),
        Mean_Predicted_Risk=(
            "Predicted Probability",
            "mean"
        ),
        Actual_Delay_Rate=(
            "Actual",
            "mean"
        )
    )
    .reset_index()
)

calibration_summary[
    "Mean_Predicted_Risk"
] *= 100

calibration_summary[
    "Actual_Delay_Rate"
] *= 100


print("\n" + "=" * 70)
print("CALIBRATION BY RISK DECILE")
print("=" * 70)

display(
    calibration_summary.round(2)
)

MODEL PROBABILITY CALIBRATION
Brier Score: 0.2196

CALIBRATION BY RISK DECILE


,Risk Bin,Flights,Mean_Predicted_Risk,Actual_Delay_Rate
0,"(0.0492, 0.251]",8091,19.60,19.92
1,"(0.251, 0.319]",8091,28.63,25.72
2,"(0.319, 0.376]",8092,34.83,32.16
3,"(0.376, 0.427]",8089,40.15,35.84
4,"(0.427, 0.479]",8091,45.30,40.37
5,"(0.479, 0.534]",8091,50.57,44.14
6,"(0.534, 0.596]",8091,56.41,48.78
7,"(0.596, 0.668]",8090,63.04,53.84
8,"(0.668, 0.761]",8091,71.24,64.71
9,"(0.761, 0.972]",8091,82.54,79.97


In [ ]:
# ============================================================
# PIECE 15 — CATBOOST CLASSIFIER
# ============================================================

# Install catboost if not present
!pip install catboost -q

from catboost import CatBoostClassifier
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# ------------------------------------------------------------
# Features
# ------------------------------------------------------------

catboost_categorical_features = [
    "Airline",
    "Flight",
    "AirportFrom",
    "AirportTo",
    "DayOfWeek"
]

catboost_numerical_features = [
    "Time",
    "Length"
]


# ------------------------------------------------------------
# Make copies
# ------------------------------------------------------------

X_train_cb = X_train.copy()
X_val_cb = X_validation.copy()  # FIX: Corrected from X_val to X_validation
X_test_cb = X_test.copy()


# CatBoost requires categorical columns to be strings
for feature in catboost_categorical_features:

    X_train_cb[feature] = (
        X_train_cb[feature].astype(str)
    )

    X_val_cb[feature] = (
        X_val_cb[feature].astype(str)
    )

    X_test_cb[feature] = (
        X_test_cb[feature].astype(str)
    )


# ------------------------------------------------------------
# CatBoost
# ------------------------------------------------------------

catboost_model = CatBoostClassifier(
    iterations=500,
    depth=7,
    learning_rate=0.08,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    thread_count=-1,
    l2_leaf_reg=5
)


# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

print("=" * 70)
print("TRAINING CATBOOST")
print("=" * 70)

catboost_model.fit(
    X_train_cb,
    y_train,
    cat_features=catboost_categorical_features,
    eval_set=(X_val_cb, y_validation),  # FIX: Corrected y_val to y_validation
    early_stopping_rounds=50,
    verbose=100
)

TRAINING CATBOOST
0:	test: 0.6891984	best: 0.6891984 (0)	total: 1.58s	remaining: 13m 10s
100:	test: 0.7243200	best: 0.7243200 (100)	total: 1m 19s	remaining: 5m 14s
200:	test: 0.7268564	best: 0.7268564 (200)	total: 2m 18s	remaining: 3m 26s
300:	test: 0.7286213	best: 0.7286213 (300)	total: 3m 18s	remaining: 2m 11s
400:	test: 0.7295714	best: 0.7295714 (400)	total: 4m 18s	remaining: 1m 3s
499:	test: 0.7300403	best: 0.7300652 (495)	total: 5m 18s	remaining: 0us

bestTest = 0.7300651977
bestIteration = 495

Shrink model to first 496 iterations.


CatBoostClassifier(depth=7, eval_metric='AUC', iterations=500, l2_leaf_reg=5, learning_rate=0.08, loss_function='Logloss', random_seed=42, verbose=100)

In [ ]:
# ============================================================
# CATBOOST — VALIDATION PERFORMANCE
# ============================================================

cb_val_prob = catboost_model.predict_proba(
    X_val_cb
)[:, 1]

cb_val_pred = (
    cb_val_prob >= 0.50
).astype(int)


cb_val_pr_auc = average_precision_score(
    y_validation,
    cb_val_prob
)

cb_val_precision = precision_score(
    y_validation,
    cb_val_pred
)

cb_val_recall = recall_score(
    y_validation,
    cb_val_pred
)

cb_val_f1 = f1_score(
    y_validation,
    cb_val_pred
)


print("=" * 70)
print("CATBOOST — VALIDATION")
print("=" * 70)

print(
    f"PR-AUC    : {cb_val_pr_auc:.4f}"
)

print(
    f"Precision : {cb_val_precision:.4f}"
)

print(
    f"Recall    : {cb_val_recall:.4f}"
)

print(
    f"F1 Score  : {cb_val_f1:.4f}"
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_validation,
        cb_val_pred
    )
)

CATBOOST — VALIDATION
PR-AUC    : 0.6987
Precision : 0.6785
Recall    : 0.5172
F1 Score  : 0.5870

Confusion Matrix:
[[36036  8832]
 [17398 18641]]


In [ ]:
# ============================================================
# PIECE 16 — CATBOOST FINAL TEST EVALUATION
# ============================================================

cb_test_prob = catboost_model.predict_proba(
    X_test_cb
)[:, 1]

cb_test_pred = (
    cb_test_prob >= 0.50
).astype(int)

print("=" * 70)
print("CATBOOST — FINAL TEST")
print("=" * 70)

print(f"ROC-AUC   : {roc_auc_score(y_test, cb_test_prob):.4f}")
print(f"PR-AUC    : {average_precision_score(y_test, cb_test_prob):.4f}")
print(f"Precision : {precision_score(y_test, cb_test_pred):.4f}")
print(f"Recall    : {recall_score(y_test, cb_test_pred):.4f}")
print(f"F1 Score  : {f1_score(y_test, cb_test_pred):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, cb_test_pred))

CATBOOST — FINAL TEST
ROC-AUC   : 0.7320
PR-AUC    : 0.7006
Precision : 0.6791
Recall    : 0.5170
F1 Score  : 0.5870

Confusion Matrix:
[[36063  8805]
 [17408 18632]]


In [ ]:
# ============================================================
# PIECE 16 — CATBOOST THRESHOLD OPTIMIZATION
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)


# ------------------------------------------------------------
# Validation probabilities
# ------------------------------------------------------------

cb_val_prob = catboost_model.predict_proba(
    X_val_cb
)[:, 1]


# PR-AUC does not depend on classification threshold
cb_val_pr_auc = average_precision_score(
    y_validation,
    cb_val_prob
)


# ------------------------------------------------------------
# Test multiple thresholds
# ------------------------------------------------------------

thresholds = np.arange(
    0.20,
    0.81,
    0.01
)

threshold_results = []


for threshold in thresholds:

    predictions = (
        cb_val_prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_validation,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_validation,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_validation,
        predictions,
        zero_division=0
    )

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })


threshold_df = pd.DataFrame(
    threshold_results
)


# ------------------------------------------------------------
# Best F1 threshold
# ------------------------------------------------------------

best_row = threshold_df.loc[
    threshold_df["F1"].idxmax()
]


best_threshold = best_row[
    "Threshold"
]


print("=" * 70)
print("CATBOOST THRESHOLD OPTIMIZATION")
print("=" * 70)

print(
    f"Validation PR-AUC : {cb_val_pr_auc:.4f}"
)

print(
    f"Best Threshold    : {best_threshold:.2f}"
)

print(
    f"Precision         : {best_row['Precision']:.4f}"
)

print(
    f"Recall            : {best_row['Recall']:.4f}"
)

print(
    f"F1 Score          : {best_row['F1']:.4f}"
)


# ------------------------------------------------------------
# Show top thresholds
# ------------------------------------------------------------

print("\nTop threshold configurations:")

display(
    threshold_df
    .sort_values("F1", ascending=False)
    .head(10)
    .round(4)
)

CATBOOST THRESHOLD OPTIMIZATION
Validation PR-AUC : 0.6987
Best Threshold    : 0.34
Precision         : 0.5499
Recall            : 0.8165
F1 Score          : 0.6572

Top threshold configurations:


,Threshold,Precision,Recall,F1
14,0.34,0.5499,0.8165,0.6572
13,0.33,0.5433,0.8311,0.6571
15,0.35,0.5568,0.8012,0.6570
12,0.32,0.5363,0.8466,0.6566
16,0.36,0.5641,0.7846,0.6563
17,0.37,0.5720,0.7684,0.6558
11,0.31,0.5297,0.8605,0.6557
10,0.30,0.5237,0.8734,0.6548
9,0.29,0.5184,0.8863,0.6541
18,0.38,0.5794,0.7495,0.6536


In [ ]:
# ============================================================
# PIECE 17 — CATBOOST FINAL TEST EVALUATION
# LOCKED THRESHOLD = 0.34
# ============================================================

from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# ------------------------------------------------------------
# Locked threshold selected from validation
# ------------------------------------------------------------

FINAL_THRESHOLD = best_threshold

print(
    f"Locked classification threshold: "
    f"{FINAL_THRESHOLD:.2f}"
)


# ------------------------------------------------------------
# Test probabilities
# ------------------------------------------------------------

cb_test_prob = catboost_model.predict_proba(
    X_test_cb
)[:, 1]


# ------------------------------------------------------------
# Test predictions using LOCKED threshold
# ------------------------------------------------------------

cb_test_pred = (
    cb_test_prob >= FINAL_THRESHOLD
).astype(int)


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

cb_test_pr_auc = average_precision_score(
    y_test,
    cb_test_prob
)

cb_test_precision = precision_score(
    y_test,
    cb_test_pred,
    zero_division=0
)

cb_test_recall = recall_score(
    y_test,
    cb_test_pred,
    zero_division=0
)

cb_test_f1 = f1_score(
    y_test,
    cb_test_pred,
    zero_division=0
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CATBOOST — FINAL TEST")
print("=" * 70)

print(
    f"PR-AUC    : {cb_test_pr_auc:.4f}"
)

print(
    f"Precision : {cb_test_precision:.4f}"
)

print(
    f"Recall    : {cb_test_recall:.4f}"
)

print(
    f"F1 Score  : {cb_test_f1:.4f}"
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        cb_test_pred
    )
)

Locked classification threshold: 0.34

CATBOOST — FINAL TEST
PR-AUC    : 0.7006
Precision : 0.5501
Recall    : 0.8201
F1 Score  : 0.6585

Confusion Matrix:
[[20696 24172]
 [ 6485 29555]]


In [ ]:
# ============================================================
# PIECE 18 — CATBOOST RISK SEGMENTATION
# ============================================================

catboost_risk_analysis = X_test_cb.copy()

catboost_risk_analysis["Actual Delay"] = (
    y_test.values
)

catboost_risk_analysis["Predicted Risk"] = (
    cb_test_prob
)


# ------------------------------------------------------------
# Risk segments
# ------------------------------------------------------------

catboost_risk_analysis["Risk Segment"] = pd.cut(
    catboost_risk_analysis["Predicted Risk"],
    bins=[
        -np.inf,
        0.30,
        0.50,
        0.70,
        np.inf
    ],
    labels=[
        "Low",
        "Moderate",
        "High",
        "Very High"
    ]
)


# ------------------------------------------------------------
# Segment statistics
# ------------------------------------------------------------

catboost_risk_summary = (
    catboost_risk_analysis
    .groupby(
        "Risk Segment",
        observed=True
    )
    .agg(
        Flights=("Actual Delay", "size"),
        Actual_Delay_Rate=(
            "Actual Delay",
            "mean"
        ),
        Average_Predicted_Risk=(
            "Predicted Risk",
            "mean"
        )
    )
    .reset_index()
)


catboost_risk_summary[
    "Actual_Delay_Rate"
] *= 100

catboost_risk_summary[
    "Average_Predicted_Risk"
] *= 100


print("=" * 70)
print("CATBOOST — FLIGHT DELAY RISK SEGMENTATION")
print("=" * 70)

display(
    catboost_risk_summary.round(2)
)

CATBOOST — FLIGHT DELAY RISK SEGMENTATION


,Risk Segment,Flights,Actual_Delay_Rate,Average_Predicted_Risk
0,Low,20748,21.42,21.77
1,Moderate,32723,39.62,39.83
2,High,17202,59.20,58.22
3,Very High,10235,82.54,82.07


In [ ]:
# ============================================================
# PIECE 19 — CATBOOST PROBABILITY CALIBRATION
# ============================================================

from sklearn.metrics import brier_score_loss


catboost_brier_score = brier_score_loss(
    y_test,
    cb_test_prob
)


print("=" * 70)
print("CATBOOST — PROBABILITY CALIBRATION")
print("=" * 70)

print(
    f"Brier Score: {catboost_brier_score:.4f}"
)


# ------------------------------------------------------------
# Calibration by decile
# ------------------------------------------------------------

catboost_calibration = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted Probability": cb_test_prob
})


catboost_calibration["Risk Bin"] = pd.qcut(
    catboost_calibration[
        "Predicted Probability"
    ],
    q=10,
    duplicates="drop"
)


catboost_calibration_summary = (
    catboost_calibration
    .groupby(
        "Risk Bin",
        observed=True
    )
    .agg(
        Flights=("Actual", "size"),
        Mean_Predicted_Risk=(
            "Predicted Probability",
            "mean"
        ),
        Actual_Delay_Rate=(
            "Actual",
            "mean"
        )
    )
    .reset_index()
)


catboost_calibration_summary[
    "Mean_Predicted_Risk"
] *= 100

catboost_calibration_summary[
    "Actual_Delay_Rate"
] *= 100


print("\n" + "=" * 70)
print("CATBOOST — CALIBRATION BY RISK DECILE")
print("=" * 70)

display(
    catboost_calibration_summary.round(2)
)

CATBOOST — PROBABILITY CALIBRATION
Brier Score: 0.2057

CATBOOST — CALIBRATION BY RISK DECILE


,Risk Bin,Flights,Mean_Predicted_Risk,Actual_Delay_Rate
0,"(0.0438, 0.206]",8091,15.90,15.14
1,"(0.206, 0.269]",8091,23.87,24.36
2,"(0.269, 0.322]",8091,29.61,28.96
3,"(0.322, 0.371]",8091,34.66,33.69
4,"(0.371, 0.417]",8090,39.36,38.26
5,"(0.417, 0.466]",8091,44.11,44.35
6,"(0.466, 0.524]",8090,49.43,51.00
7,"(0.524, 0.604]",8091,56.14,56.66
8,"(0.604, 0.748]",8091,66.80,68.26
9,"(0.748, 0.977]",8091,84.64,84.77


In [ ]:
# ============================================================
# PIECE 20 — CATBOOST FEATURE IMPORTANCE
# ============================================================

# ------------------------------------------------------------
# 1. Extract native CatBoost feature importance
# ------------------------------------------------------------

catboost_importance = catboost_model.get_feature_importance(
    type="FeatureImportance"
)

feature_importance_df = pd.DataFrame({
    "Feature": X_train_cb.columns,
    "Importance": catboost_importance
})


# ------------------------------------------------------------
# 2. Sort by importance
# ------------------------------------------------------------

feature_importance_df = (
    feature_importance_df
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 3. Display all features
# ------------------------------------------------------------

print("=" * 70)
print("CATBOOST — FEATURE IMPORTANCE")
print("=" * 70)

display(
    feature_importance_df.round(4)
)

CATBOOST — FEATURE IMPORTANCE


,Feature,Importance
0,Airline,25.9022
1,Time,19.4826
2,AirportFrom,15.4356
3,Flight,11.3988
4,AirportTo,10.9989
5,DayOfWeek,10.1613
6,Length,6.6206


In [ ]:
# ============================================================
# OPERATIONAL FEATURE IMPORTANCE
# ============================================================

operational_importance_df = (
    feature_importance_df[
        feature_importance_df["Feature"] != "Flight"
    ]
    .reset_index(drop=True)
)


print("=" * 70)
print("CATBOOST — OPERATIONAL FEATURE IMPORTANCE")
print("=" * 70)

display(
    operational_importance_df.round(4)
)

CATBOOST — OPERATIONAL FEATURE IMPORTANCE


,Feature,Importance
0,Airline,25.9022
1,Time,19.4826
2,AirportFrom,15.4356
3,AirportTo,10.9989
4,DayOfWeek,10.1613
5,Length,6.6206


In [ ]:
# ============================================================
# RELATIVE FEATURE IMPORTANCE
# ============================================================

operational_importance_df[
    "Importance (%)"
] = (
    operational_importance_df["Importance"]
    /
    operational_importance_df["Importance"].sum()
    * 100
)


operational_importance_df = (
    operational_importance_df
    .sort_values(
        "Importance (%)",
        ascending=False
    )
    .reset_index(drop=True)
)


print("=" * 70)
print("CATBOOST — RELATIVE OPERATIONAL IMPORTANCE")
print("=" * 70)

display(
    operational_importance_df.round(2)
)

CATBOOST — RELATIVE OPERATIONAL IMPORTANCE


,Feature,Importance,Importance (%)
0,Airline,25.90,29.23
1,Time,19.48,21.99
2,AirportFrom,15.44,17.42
3,AirportTo,11.00,12.41
4,DayOfWeek,10.16,11.47
5,Length,6.62,7.47
